In [ ]:
sheets = load_sheets(config.SHEETS_RAW_DIR)
print("листов:", len(sheets))
print({k: v.shape for k, v in list(sheets.items())[:3]}, "...")

листов: 17
{'CLIMATE': (1089, 109), 'Description': (31, 3), 'EMISSIONS': (1089, 590)} ...


In [ ]:
cleaned, rows = {}, []
for name, d in sheets.items():
    if name in config.SERVICE_SHEETS:
        continue
    cleaned[name], st = clean_sheet(d, config.KEY)
    rows.append({"лист": name, **st})
report = pd.DataFrame(rows)
report

,лист,строк_было,столбцов_было,служебных_удалено,кодировка_исправлено,обрезанных_пределов,текст_в_число,дублей_удалено,пустых_удалено,пустые_столбцы,строк_стало,столбцов_стало
0,CLIMATE,1089,109,0,0,0,0,0,0,[],1089,109
1,EMISSIONS,1089,590,0,183,0,5,0,0,[],1089,590
2,EXPOSURE,1089,1066,3,0,0,0,0,0,[],1089,1063
3,GENERAL_CHARACTERISTICS,1089,13,0,530,0,0,0,0,[],1089,13
4,GEOGRAPHY,1089,37,0,347,0,0,0,0,[],1089,37
5,GHSL,1089,551,0,0,0,0,0,0,[],1089,551
6,GREENNESS,1089,63,0,183,0,0,0,0,[],1089,63
7,HAZARD_RISK,1089,118,0,0,0,0,0,7,"[HZ_CEV_WLF_2005, HZ_CEV_EAR_2005, HZ_CEV_WLF_...",1089,111
8,HEALTH,1089,17,0,183,0,0,0,0,[],1089,17
9,INFRASTRUCTURES,1089,17,0,183,0,0,0,0,[],1089,17


In [ ]:
df, _ = build_clean(cleaned, config.KEY, sheets["Index"])
print("после соединения:", df.shape)

после соединения: (1089, 2619)


In [ ]:
index_df = sheets["Index"]
units = column_units([c.upper() for c in df.columns], index_df)
units = {k.lower(): v for k, v in units.items()}
typed, plan = apply_schema(df, units, config.FLOAT32_UNITS, config.INT16_UNITS, config.INT32_UNITS)
print("типизировано:", typed.shape, "| типов назначено:", len(plan))

типизировано: (1089, 2619) | типов назначено: 1407


In [ ]:
lines = (Path("..") / "src" / "pipeline.py").read_text(encoding="utf-8").splitlines()
print("\n".join(f"{i} {l}" for i, l in enumerate(lines[114:118], 115)))

115     typed, plan = apply_schema(df, units, config.FLOAT32_UNITS, config.INT16_UNITS,
116                                config.INT32_UNITS, config.CATEGORY_MAX_SHARE)
117     out_path.parent.mkdir(parents=True, exist_ok=True)
118     typed.to_parquet(out_path, compression="zstd")


In [ ]:
assert_frame_equal(df, pd.read_parquet(config.INTERIM_DIR / "clean_untyped.parquet"))
assert_frame_equal(typed, pd.read_parquet(config.PROCESSED_DIR / "clean.parquet"), check_dtype=True)
print("блокнот и сценарий совпадают:", typed.shape)

блокнот и сценарий совпадают: (1089, 2619)
